<a href="https://colab.research.google.com/github/matjesg/deepflash2/blob/colab-test-modernize/colab_eval_encoders.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# deepflash2 — encoder eval: default vs. new (ConvNeXt V2)

Trains one K-fold split on the sample data and compares **validation Dice** for:
- **default**: `tu-convnext_tiny` (current `config.py` default)
- **new**: `tu-convnextv2_tiny.fcmae_ft_in22k_in1k` (ConvNeXt V2, fcmae-pretrained)

Both use identical settings and the same fold (`random_state=42`), so only the encoder differs.

**Before running:** `Runtime` → `Change runtime type` → **GPU** (ConvNeXt trains on CUDA; it can't train on Apple MPS due to an upstream timm issue). Then `Runtime` → `Run all`.

> Note: this is a *quick* comparison on 5 sample images / one fold — indicative, not a rigorous benchmark. Bump `N_EPOCHS` and/or average over folds for a real evaluation.

In [ ]:
#@title 1. Install deepflash2 (modernization branch) + check GPU
!pip install -Uqq "git+https://github.com/matjesg/deepflash2.git@feat/modernize-stack"
from IPython.display import clear_output
import torch, deepflash2
clear_output(wait=True)
print('deepflash2', deepflash2.__version__, '| torch', torch.__version__, '| cuda:', torch.cuda.is_available())
assert torch.cuda.is_available(), 'Select a GPU runtime: Runtime -> Change runtime type -> GPU'

In [ ]:
#@title 2. Download sample training data (5 image/mask pairs)
from deepflash2.utils import download_sample_data
download_sample_data('https://github.com/matjesg/deepflash2/releases/download/sample_data/',
                     'wue1_cFOS_small.zip', 'deepflash2_sample_data', extract=True)

In [ ]:
#@title 3. Train fold 1 for each encoder and compare validation Dice
import warnings; warnings.filterwarnings('ignore')
import gc, torch, pandas as pd
from deepflash2.config import Config
from deepflash2.learner import EnsembleLearner

DATA = 'deepflash2_sample_data'
N_EPOCHS = 20  #@param {type:'integer'}  bump for a more thorough comparison

def eval_encoder(enc):
    cfg = Config(encoder_name=enc, n_epochs=N_EPOCHS)  # random_state=42 -> identical fold split
    el = EnsembleLearner(image_dir='images', mask_dir='masks', path=DATA, config=cfg)
    el.fit(1)  # train fold 1
    rec = el.recorder[1]
    di = list(rec.metric_names).index('dice') - 1  # values rows exclude the 'epoch' column
    best = max(float(v[di]) for v in rec.values)
    del el; gc.collect()
    if torch.cuda.is_available(): torch.cuda.empty_cache()
    return best

rows = []
for setting, enc in [('default (convnext_tiny)', 'tu-convnext_tiny'),
                     ('new (convnextv2_tiny fcmae)', 'tu-convnextv2_tiny.fcmae_ft_in22k_in1k')]:
    print(f'\n===== training {setting}: {enc} =====')
    rows.append({'setting': setting, 'encoder': enc, 'best_val_dice': round(eval_encoder(enc), 4)})

df = pd.DataFrame(rows)
print(f'\n===== Validation Dice (fold 1, identical split, N_EPOCHS={N_EPOCHS}) =====')
print(df.to_string(index=False))
df